# SegFormer-B0 Training
Uses ImageNet-only `nvidia/mit-b0` encoder pretraining. Do not use the old ADE20K smoke checkpoint for E2.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
PROJECT = Path("/content/TTTN")
assert PROJECT.exists(), "Upload/unzip the project to /content/TTTN first"
assert (PROJECT / "data" / "3cad_ani").exists(), "Dataset missing at /content/TTTN/data/3cad_ani"
%cd /content/TTTN


In [ ]:
!pip install -q -r requirements/ml-kaggle.txt
!python scripts/verification/check_protocol.py


## Main training


In [ ]:
!python scripts/training/train_segformer.py \
  --epochs 50 \
  --batch-size 2 \
  --grad-accum 2 \
  --augmentation photometric \
  --run-name main_seed42


## Validation — select threshold


In [ ]:
!python scripts/evaluation/evaluate_model.py \
  --model segformer \
  --checkpoint results/segformer_b0/main_seed42/checkpoints/best.pt \
  --split val --warmup-batches 5


## Final Test


In [ ]:
!python scripts/evaluation/evaluate_model.py \
  --model segformer \
  --checkpoint results/segformer_b0/main_seed42/checkpoints/best.pt \
  --split test --warmup-batches 5


## Inspect evidence


In [ ]:
from IPython.display import display, Image
import pandas as pd
base="results/segformer_b0/main_seed42"
for f in [
    "curves/learning_curve_loss.png",
    "curves/learning_curve_dice.png",
    "curves/loss_components.png",
    "test/figures/image_roc_curve.png",
    "test/figures/image_pr_curve.png",
    "test/figures/confusion_matrix.png",
]: display(Image(f"{base}/{f}"))
display(pd.read_csv(f"{base}/test/main_metrics.csv"))
